In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [14]:
df = pd.read_csv(r"C:\Users\91735\Downloads\data.csv")

print(df.head())
print(df.info())

                  date      price  bedrooms  bathrooms  sqft_living  sqft_lot  \
0  2014-05-02 00:00:00   313000.0       3.0       1.50         1340      7912   
1  2014-05-02 00:00:00  2384000.0       5.0       2.50         3650      9050   
2  2014-05-02 00:00:00   342000.0       3.0       2.00         1930     11947   
3  2014-05-02 00:00:00   420000.0       3.0       2.25         2000      8030   
4  2014-05-02 00:00:00   550000.0       4.0       2.50         1940     10500   

   floors  waterfront  view  condition  sqft_above  sqft_basement  yr_built  \
0     1.5           0     0          3        1340              0      1955   
1     2.0           0     4          5        3370            280      1921   
2     1.0           0     0          4        1930              0      1966   
3     1.0           0     0          4        1000           1000      1963   
4     1.0           0     0          4        1140            800      1976   

   yr_renovated                    str

In [15]:
print(df.isnull().sum())

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64


In [16]:
df = df.dropna()

In [17]:
encoder = LabelEncoder()

categorical_columns = [
    'date',
    'street',
    'city',
    'statezip',
    'country'
]

for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])

In [18]:
X = df.drop("price", axis=1)
y = df["price"]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [20]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [21]:
model = Sequential()

model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))

c:\Users\91735\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [23]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 448286883840.0000 - mae: 546844.8125 - val_loss: 401456496640.0000 - val_mae: 536765.3125
Epoch 2/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 447957860352.0000 - mae: 546610.1875 - val_loss: 400673734656.0000 - val_mae: 536169.8125
Epoch 3/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 445547905024.0000 - mae: 545012.4375 - val_loss: 396561678336.0000 - val_mae: 533153.5000
Epoch 4/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 436944306176.0000 - mae: 539365.8125 - val_loss: 384754319360.0000 - val_mae: 524546.9375
Epoch 5/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 416466206720.0000 - mae: 525902.1250 - val_loss: 360362967040.0000 - val_mae: 506330.9062
Epoch 6/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 378409287680.0000 - mae: 500547.8125 - val_loss: 319123193856.0000 - val_mae: 473612.3438
Epoch 7/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 321917157376.0000 - mae: 458277.7812 - val_los

In [24]:
loss, mae = model.evaluate(X_test, y_test)

print("Test Loss :", loss)
print("Test MAE :", mae)

29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 979549421568.0000 - mae: 199331.9062
Test Loss : 979549421568.0
Test MAE : 199331.90625


In [25]:
y_pred = model.predict(X_test)

29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [26]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE :", rmse)

RMSE : 989721.9019548113


In [27]:
result = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred.flatten()
})

print(result.head(10))

   Actual Price  Predicted Price
0  5.440000e+05     3.711760e+05
1  0.000000e+00     3.503128e+05
2  1.712500e+06     1.073688e+06
3  3.650000e+05     5.221775e+05
4  2.750000e+05     3.405906e+05
5  6.250000e+05     5.722204e+05
6  4.530000e+05     4.725072e+05
7  3.000000e+05     4.190008e+05
8  4.179857e+05     4.272007e+05
9  6.725000e+05     4.733881e+05


In [28]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,333 (44.27 KB)

 Trainable params: 3,777 (14.75 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,556 (29.52 KB)